# XGBoost

In [ ]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.multioutput import MultiOutputClassifier 
from sklearn.metrics import f1_score

import re #mi serve per pulire


from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Lavoro su singola fold

In [ ]:
def fit_single_fold(train_idx, test_idx, features, target, groups, n_est, max_d, lr, sub, col, mcw, gam, alpha, lam):
  # ========== DEBUGGING: Stampo indici train/test  ==========

  """print("?"*50 + "\nDebug\n" + "?"*50)
  print(f"\nFold {fold} - File: {csv_name}")
  print(f"  Train indice: {train_index[:10]})")
  print(f"  Test indice: {test_index[:10]})")
  print(f"  Train gruppo (Patient IDs): {groups.iloc[train_index].unique()}")
  print(f"  Test gruppo (Patient IDs): {groups.iloc[test_index].unique()}")
  print("?"*100)"""
  # ==========================================================

  # Suddivido i dati in set di training e di test per la fold corrente
  X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
  y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

  rf = xgb.XGBClassifier(
        n_estimators= n_est,              
        max_depth= max_d,                
        learning_rate= lr,            
        subsample= sub,            
        colsample_bytree= col,          
        min_child_weight= mcw,         
        gamma= gam,                     
        reg_alpha= alpha,              
        reg_lambda= lam 
    )
  
  multi_output_xgb = MultiOutputClassifier(rf)

  # Addestro il modello sul train set di questa fold
  multi_output_xgb.fit(X_train, y_train)

  # Predico il target sul test set di questa fold
  y_pred = multi_output_xgb.predict(X_test)

  # DEBUG
  #score = f1_score(y_test, y_pred, average="micro")
  #print(f"Fold {fold} - {max_iter=}, {max_depth=}, {min_samples_leaf=}, Score={score:.4f}")

  return f1_score(y_test, y_pred, average="micro")

# Training


In [ ]:
def training(file_path, csv_name):
    # Vado a leggere il csv
    df = pd.read_csv(file_path)

    # Definisco le colonne target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']

    # Vado a rimuovere le lesioni (righe) non valide
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo tutto in valori binari per "facilitare" il lavoro
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)  # Soglia clinica comune per KI67

    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']

    """
     Preparo le feature (X) e i target (y) per il modello
    """
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]

    # 'groups' contiene l'ID del paziente per ogni lesione.
    # Mi serve per fare la cross-validation a gruppo
    groups = df_validi['Patient ID']

    # Riempip a Nan se è rimasto vuoto
    features = features.fillna(features.mean())

    """ Dovrei pulire il nome delle colonne per farlo andare """
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]


    # Imposto la strategia di cross-validation.
    # GroupKFold assicura che le lesioni dello stesso paziente non vengano mai divise tra training set e test set
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)

    # Lista vuota per collezionare i punteggi di performance di ogni fold.
    scores = []

    # Definisco gli iperparametri
    iperparametri = {
        'n_estimators': [100],              # Numero di alberi
        'max_depth': [3, 5],                # Profondità massima degli alberi
        'learning_rate': [0.05],            # Tasso di apprendimento
        'subsample': [0.6, 1.0],            # Frazione di campioni per albero
        'colsample_bytree': [0.6],          # Frazione di feature per albero
        'min_child_weight': [1, 3],         # Somma minima dei pesi nelle foglie
        'gamma': [0.1],                     # Riduzione minima loss per split
        'reg_alpha': [0.1, 1],              # Regolarizzazione L1
        'reg_lambda': [1, 2]           # Regolarizzazione L2
    }

    scores = []

    # Qui devo farci i for manuali
    for n_est in iperparametri['n_estimators']:
        for max_d in iperparametri['max_depth']:
            for lr in iperparametri['learning_rate']:
                for sub in iperparametri['subsample']:
                    for col in iperparametri['colsample_bytree']:
                        for mcw in iperparametri['min_child_weight']:
                            for gam in iperparametri['gamma']:
                                for alpha in iperparametri['reg_alpha']:
                                    for lam in iperparametri['reg_lambda']:
                                        fold_scores = Parallel(n_jobs=-1)(delayed(fit_single_fold)(train_idx, test_idx, features, target, groups, n_est, max_d, lr, sub, col, mcw, gam, alpha, lam) for train_idx, test_idx in cv.split(features, target, groups))
                                        
                                        # calcolo media e deviazione standard degli score su tutte le fold
                                        mean_score = np.mean(fold_scores)
                                        std_score = np.std(fold_scores)

                                        # registro i risultati per la combinazione di parametri corrente
                                        scores.append({
                                            'n_estimators': n_est,
                                            'max_depth': max_d,
                                            'learning_rate': lr,
                                            'subsample': sub,
                                            'colsample_bytree': col,
                                            'min_child_weight': mcw,
                                            'gamma': gam,
                                            'reg_alpha': alpha,
                                            'reg_lambda': lam,
                                            'mean_score': mean_score,
                                            'std_score': std_score,
                                            'fold_scores': fold_scores
                                        })

    return scores

# Lettura dei file

In [ ]:
#TODO: GIA MODIFICATO


results_per_dataset = {}

for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Stampo i risultati per ogni dataset
for name, metrics_list in results_per_dataset.items():

    # Stampo solo i migliori
    best_result = max(metrics_list, key=lambda x: x['mean_score'])
    print(f"\nDataset: {name}:")
    print(f"  n_estimators: {best_result['n_estimators']}")
    print(f"  max_depth: {best_result['max_depth']}")
    print(f"  learning_rate: {best_result['learning_rate']}")
    print(f"  subsample: {best_result['subsample']}")
    print(f"  colsample_bytree: {best_result['colsample_bytree']}")
    print(f"  min_child_weight: {best_result['min_child_weight']}")
    print(f"  gamma: {best_result['gamma']}")
    print(f"  reg_alpha: {best_result['reg_alpha']}")
    print(f"  reg_lambda: {best_result['reg_lambda']}")
    print(f"  Mean F1-score: {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")




# va dentro al for sopra


# Lo faccio eseguire su colab, ecco l'output


Dataset: t2_medsam:
  n_estimators: 100
  max_depth: 3
  learning_rate: 0.05
  subsample: 1.0
  colsample_bytree: 0.6
  min_child_weight: 3
  gamma: 0.1
  reg_alpha: 1
  reg_lambda: 2
  Mean F1-score: 0.778 ± 0.086


Dataset: t2_preprocessed:
  n_estimators: 100
  max_depth: 3
  learning_rate: 0.05
  subsample: 0.6
  colsample_bytree: 0.6
  min_child_weight: 3
  gamma: 0.1
  reg_alpha: 0.1
  reg_lambda: 1
  Mean F1-score: 0.763 ± 0.057


Dataset: t2_original:
  n_estimators: 100
  max_depth: 3
  learning_rate: 0.05
  subsample: 0.6
  colsample_bytree: 0.6
  min_child_weight: 1
  gamma: 0.1
  reg_alpha: 1
  reg_lambda: 2
  Mean F1-score: 0.752 ± 0.058


Dataset: medsam_dynamic:
  n_estimators: 100
  max_depth: 5
  learning_rate: 0.05
  subsample: 0.6
  colsample_bytree: 0.6
  min_child_weight: 1
  gamma: 0.1
  reg_alpha: 0.1
  reg_lambda: 1
  Mean F1-score: 0.739 ± 0.049


Dataset: preprocessed_dynamic:
  n_estimators: 100
  max_depth: 5
  learning_rate: 0.05
  subsample: 0.6
  colsample_bytree: 0.6
  min_child_weight: 1
  gamma: 0.1
  reg_alpha: 1
  reg_lambda: 2
  Mean F1-score: 0.759 ± 0.054


Dataset: original_dynamic:
  n_estimators: 100
  max_depth: 3
  learning_rate: 0.05
  subsample: 0.6
  colsample_bytree: 0.6
  min_child_weight: 1
  gamma: 0.1
  reg_alpha: 1
  reg_lambda: 2
  Mean F1-score: 0.758 ± 0.057
